In [1]:
# System & IO
import os
import glob
import joblib
import random

# Data Manipulation
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Statistics & Machine Learning Baselines
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

# PyTorch & Geometric
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset

# Visualization & Progress
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm, trange

# Hardware Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

c:\Users\jaden\miniconda3\envs\flood_ml\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\jaden\miniconda3\envs\flood_ml\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\jaden\miniconda3\envs\flood_ml\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\jaden\miniconda3\envs\flood_ml\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its us

### Data Proccessing/Cleaning

In [2]:
def check_multicollinearity(df):
    # 1. Drop non-numeric and ID columns
    X = df.drop(columns=['node_idx']).select_dtypes(include=[np.number])
    
    # 2. CRITICAL: Drop rows with NaNs/Infs for the analysis
    # We replace inf with NaN then drop all NaNs
    X = X.replace([np.inf, -np.inf], np.nan).dropna()
    
    if X.empty:
        raise ValueError("After dropping NaNs, no data is left for VIF calculation!")

    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    return vif_data

def get_edge_index(path, src_mapping, dst_mapping, src_col='from_node', dst_col='to_node'):
    df = pd.read_csv(path)
    
    # Debug print to help you if it fails again
    if src_col not in df.columns or dst_col not in df.columns:
        print(f"Error in {path}! Available columns: {df.columns.tolist()}")
        raise KeyError(f"Could not find {src_col} or {dst_col}")

    src = df[src_col].map(src_mapping).values
    dst = df[dst_col].map(dst_mapping).values
    
    # Check for mapping failures
    if pd.isna(src).any() or pd.isna(dst).any():
        print(f"Warning: Some IDs in {path} weren't in your mapping. Check your static node files!")
    
    edge_index_np = np.stack([src, dst])
    return torch.from_numpy(edge_index_np).to(torch.long)

In [3]:
# 1. Load static data to establish the "Universe" of nodes
df_1d_static = pd.read_csv('../Data/Models/Model_1/train/1d_nodes_static.csv')
df_2d_static = pd.read_csv('../Data/Models/Model_1/train/2d_nodes_static.csv')

# 2. Create Mappings: {Original_ID: Integer_Index}
mapping_1d = {id: i for i, id in enumerate(df_1d_static['node_idx'])}
mapping_2d = {id: i for i, id in enumerate(df_2d_static['node_idx'])}

In [4]:
print("--- 1D Static VIF Analysis ---")
vif_1d = check_multicollinearity(df_1d_static)
print(vif_1d[vif_1d['VIF'] > 10]) # Only show the problematic ones

print("\n--- 2D Static VIF Analysis ---")
vif_2d = check_multicollinearity(df_2d_static)
print(vif_2d[vif_2d['VIF'] > 10])

--- 1D Static VIF Analysis ---
             feature           VIF
0         position_x  4.869657e+06
1         position_y  4.780871e+06
2              depth  2.756012e+11
3   invert_elevation  1.000800e+15
4  surface_elevation  1.000800e+15

--- 2D Static VIF Analysis ---
         feature           VIF
0     position_x  1.226261e+06
1     position_y  1.203382e+06
2           area  4.164334e+01
3      roughness  1.389436e+01
4  min_elevation  8.452923e+04
5      elevation  8.537107e+04


In [5]:
# 1D Pruning
df_1d_static = df_1d_static.drop(columns=['surface_elevation', 'position_x', 'position_y'])

# 2D Pruning
df_2d_static = df_2d_static.drop(columns=['min_elevation', 'position_x', 'position_y'])

In [6]:
# Check both—don't be lazy!
print("--- 1D Static VIF Analysis ---")
vif_1d = check_multicollinearity(df_1d_static)
print(vif_1d[vif_1d['VIF'] > 10]) # Only show the problematic ones

print("\n--- 2D Static VIF Analysis ---")
vif_2d = check_multicollinearity(df_2d_static)
print(vif_2d[vif_2d['VIF'] > 10])

print("\n--- Final Features ---")
print(f"1D Features: {df_1d_static.columns.tolist()}")
print(f"2D Features: {df_2d_static.columns.tolist()}")

# Ensure no NaNs are left to break the Scaler
df_1d_static = df_1d_static.fillna(0)
df_2d_static = df_2d_static.fillna(0)

--- 1D Static VIF Analysis ---
Empty DataFrame
Columns: [feature, VIF]
Index: []

--- 2D Static VIF Analysis ---
     feature        VIF
0       area  36.596482
1  roughness  13.525360
2  elevation  34.043408

--- Final Features ---
1D Features: ['node_idx', 'depth', 'invert_elevation', 'base_area']
2D Features: ['node_idx', 'area', 'roughness', 'elevation', 'aspect', 'curvature', 'flow_accumulation']


In [7]:
# Define the skeleton before the loop
edge_index_1d = get_edge_index(
    '../Data/Models/Model_1/train/1d_edge_index.csv', mapping_1d, mapping_1d
)
edge_index_2d = get_edge_index(
    '../Data/Models/Model_1/train/2d_edge_index.csv', mapping_2d, mapping_2d
)
# Use your specific column names if they differ for 1d2d
edge_index_1d2d = get_edge_index(
    '../Data/Models/Model_1/train/1d2d_connections.csv', mapping_1d, mapping_2d,
    src_col='node_1d', dst_col='node_2d' # Double-check these column names!
)

In [8]:
# def process_flood_directory(source_path, folders, output_dir, edge_indices, scalers, dataset_tag="Train"):
#     os.makedirs(output_dir, exist_ok=True)
#     s1s, s2s, s1d, s2d = scalers
#     e1, e2, e12 = edge_indices
    
#     # Stats for assignment reporting
#     stats = {"total": len(folders), "skipped_missing": 0, "skipped_physical": 0, "skipped_shape": 0, "success": 0}
    
#     static_1d_scaled = s1s.transform(df_1d_static.drop(columns=['node_idx']))
#     static_2d_scaled = s2s.transform(df_2d_static.drop(columns=['node_idx']))

#     for folder in tqdm(folders, desc=f"Processing {dataset_tag}"):
#         event_path = os.path.join(source_path, folder)
#         try:
#             df_1d_dyn = pd.read_csv(os.path.join(event_path, '1d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
#             df_2d_dyn = pd.read_csv(os.path.join(event_path, '2d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
#             num_t = len(pd.read_csv(os.path.join(event_path, 'timesteps.csv')))
            
#             # --- 1. QUALITY GATE ---
#             # Using strict mode for all internal splits to ensure high data quality
#             missing_ratio = max(df_1d_dyn.isnull().mean().mean(), df_2d_dyn.isnull().mean().mean())
#             if missing_ratio > 0.10:
#                 stats["skipped_missing"] += 1
#                 continue

#             # --- 2. PHYSICAL CONSISTENCY CHECK ---
#             numeric_1d = df_1d_dyn.select_dtypes(include=np.number).dropna(axis=1)
#             if (numeric_1d < -0.01).any().any():
#                 stats["skipped_physical"] += 1
#                 continue

#             # CLEANING
#             df_1d_dyn, df_2d_dyn = df_1d_dyn.fillna(0), df_2d_dyn.fillna(0)

#             # --- 3. TENSORIZATION & NORMALIZATION ---
#             x1d = s1d.transform(df_1d_dyn.drop(columns=['node_idx', 'timestep']))
#             x2d = s2d.transform(df_2d_dyn.drop(columns=['node_idx', 'timestep']))
            
#             x1d_3d = np.nan_to_num(x1d).reshape(num_t, num_1d_nodes, -1)
#             x2d_3d = np.nan_to_num(x2d).reshape(num_t, num_2d_nodes, -1)

#             x1s_3d = np.repeat(static_1d_scaled[np.newaxis, :, :], num_t, axis=0)
#             x2s_3d = np.repeat(static_2d_scaled[np.newaxis, :, :], num_t, axis=0)

#             # --- 4. HETERO DATA ENCAPSULATION ---
#             event_data = HeteroData()
#             event_data['node1d'].x = torch.from_numpy(np.concatenate([x1s_3d, x1d_3d], axis=-1)).float()
#             event_data['node2d'].x = torch.from_numpy(np.concatenate([x2s_3d, x2d_3d], axis=-1)).float()
            
#             event_data['node1d', 'pipe', 'node1d'].edge_index = e1
#             event_data['node2d', 'surface', 'node2d'].edge_index = e2
#             event_data['node1d', 'exchange', 'node2d'].edge_index = e12

#             # --- 5. LOG EXAMPLE SAMPLE ---
#             if stats["success"] == 0 and dataset_tag == "Train":
#                 print(f"\n--- Example Cleaned Sample ({folder}) ---")
#                 print(f"1D Tensor Shape: {event_data['node1d'].x.shape} (T, N, F)")
#                 print(f"2D Tensor Shape: {event_data['node2d'].x.shape}")
#                 print(f"Sample Feature Slice (Step 0): {event_data['node1d'].x[0, 0, :5]}")

#             torch.save(event_data, os.path.join(output_dir, f'event_{folder}.pt'))
#             stats["success"] += 1
#         except Exception as e:
#             stats["skipped_shape"] += 1
#             continue

#     print(f"\n--- {dataset_tag} Statistics ---")
#     for k, v in stats.items(): print(f"{k}: {v}")

In [9]:
def process_flood_directory(source_path, folders, output_dir, edge_indices, scalers, dataset_tag="Train"):
    os.makedirs(output_dir, exist_ok=True)
    s1s, s2s, s1d, s2d = scalers
    e1, e2, e12 = edge_indices
    
    # Stats for assignment reporting - expanded for better visibility
    stats = {
        "total": len(folders), 
        "skipped_missing_val": 0, 
        "skipped_physical_violation": 0, 
        "skipped_shape_mismatch": 0,
        "skipped_empty_graph": 0,
        "success": 0
    }
    
    # Pre-scale static data to avoid redundant computation in the loop
    static_1d_scaled = s1s.transform(df_1d_static.drop(columns=['node_idx']))
    static_2d_scaled = s2s.transform(df_2d_static.drop(columns=['node_idx']))

    for folder in tqdm(folders, desc=f"Processing {dataset_tag}"):
        event_path = os.path.join(source_path, folder)
        try:
            # Load and sort to ensure temporal/spatial alignment
            df_1d_dyn = pd.read_csv(os.path.join(event_path, '1d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
            df_2d_dyn = pd.read_csv(os.path.join(event_path, '2d_nodes_dynamic_all.csv')).sort_values(['timestep', 'node_idx'])
            df_time = pd.read_csv(os.path.join(event_path, 'timesteps.csv'))
            
            num_t = len(df_time)
            num_1d = static_1d_scaled.shape[0]
            num_2d = static_2d_scaled.shape[0]

            # --- 1. DATA INTEGRITY & SHAPE CHECK ---
            # Verify that we have the exact number of rows expected (T * N)
            expected_1d_rows = num_t * num_1d
            expected_2d_rows = num_t * num_2d
            
            if len(df_1d_dyn) != expected_1d_rows or len(df_2d_dyn) != expected_2d_rows:
                stats["skipped_shape_mismatch"] += 1
                continue

            # --- 2. QUALITY GATE (Missing Values) ---
            missing_ratio = max(df_1d_dyn.isnull().mean().max(), df_2d_dyn.isnull().mean().max())
            if missing_ratio > 0.10:
                stats["skipped_missing_val"] += 1
                continue

            # --- 3. PHYSICAL CONSISTENCY (e.g., Water depth cannot be negative) ---
            numeric_1d = df_1d_dyn.select_dtypes(include=np.number)
            if (numeric_1d < -1e-3).any().any(): # Small epsilon for float noise
                stats["skipped_physical_violation"] += 1
                continue

            # Impute remaining small holes
            df_1d_dyn = df_1d_dyn.fillna(0)
            df_2d_dyn = df_2d_dyn.fillna(0)

            # --- 4. TENSORIZATION & NORMALIZATION ---
            x1d = s1d.transform(df_1d_dyn.drop(columns=['node_idx', 'timestep']))
            x2d = s2d.transform(df_2d_dyn.drop(columns=['node_idx', 'timestep']))
            
            # Reshape to (Timesteps, Nodes, Features)
            x1d_3d = x1d.reshape(num_t, num_1d, -1)
            x2d_3d = x2d.reshape(num_t, num_2d, -1)

            # Broadcast static features across all timesteps
            x1s_3d = np.repeat(static_1d_scaled[np.newaxis, :, :], num_t, axis=0)
            x2s_3d = np.repeat(static_2d_scaled[np.newaxis, :, :], num_t, axis=0)

            # --- 5. HETERO DATA ENCAPSULATION ---
            event_data = HeteroData()
            
            # Combine static and dynamic features along the feature dimension (axis=-1)
            event_data['node1d'].x = torch.from_numpy(np.concatenate([x1s_3d, x1d_3d], axis=-1)).float()
            event_data['node2d'].x = torch.from_numpy(np.concatenate([x2s_3d, x2d_3d], axis=-1)).float()
            
            # Store topology
            event_data['node1d', 'pipe', 'node1d'].edge_index = e1
            event_data['node2d', 'surface', 'node2d'].edge_index = e2
            event_data['node1d', 'exchange', 'node2d'].edge_index = e12
            
            # Metadata for easy downstream filtering
            event_data.num_timesteps = num_t
            event_data.event_id = folder

            # --- 6. SAMPLE LOGGING ---
            if stats["success"] == 0 and dataset_tag == "Train":
                print(f"\n--- {dataset_tag} Sample Verified: {folder} ---")
                print(f"Nodes: 1D={num_1d}, 2D={num_2d} | Timesteps={num_t}")
                print(f"Edges: Pipe={e1.shape[1]}, Surface={e2.shape[1]}, Exchange={e12.shape[1]}")
                print(f"Feature Vector Size: 1D={event_data['node1d'].x.shape[-1]}")

            torch.save(event_data, os.path.join(output_dir, f'event_{folder}.pt'))
            stats["success"] += 1
            
        except FileNotFoundError:
            stats["skipped_missing_val"] += 1
        except ValueError as ve:
            # Catches reshape errors specifically
            stats["skipped_shape_mismatch"] += 1
        except Exception as e:
            print(f"Unexpected error in {folder}: {e}")
            stats["skipped_empty_graph"] += 1

    print(f"\n--- {dataset_tag} Final Report ---")
    for k, v in stats.items():
        print(f"{k.replace('_', ' ').title()}: {v}")

In [10]:
# --- 1. SETUP & DATA LOADING ---
# We are now operating exclusively within the train_path for all splits
train_path = '../Data/Models/Model_1/train/'

df_1d_static = pd.read_csv(os.path.join(train_path, '1d_nodes_static.csv'))
df_2d_static = pd.read_csv(os.path.join(train_path, '2d_nodes_static.csv'))

# VIF DROPS: Removing redundant features to ensure model stability
df_1d_static = df_1d_static.drop(columns=['surface_elevation', 'position_x', 'position_y'])
df_2d_static = df_2d_static.drop(columns=['min_elevation', 'position_x', 'position_y'])

# Define node counts for tensor reshaping
num_1d_nodes = len(df_1d_static)
num_2d_nodes = len(df_2d_static)

# --- 2. SCALER INITIALIZATION & FITTING ---
scaler_1d_static = StandardScaler()
scaler_2d_static = StandardScaler()
scaler_dyn_1d = StandardScaler()
scaler_dyn_2d = StandardScaler()

print("Fitting static scalers...")
scaler_1d_static.fit(df_1d_static.drop(columns=['node_idx']))
scaler_2d_static.fit(df_2d_static.drop(columns=['node_idx']))

# SPLIT LOGIC: 80/10/10 split of the 68 healthy events
all_train_folders = [f for f in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, f)) and f != 'Processed']
random.shuffle(all_train_folders)

train_idx = int(0.8 * len(all_train_folders))
val_idx = int(0.9 * len(all_train_folders))

train_folders = all_train_folders[:train_idx]
val_folders = all_train_folders[train_idx:val_idx]
test_folders = all_train_folders[val_idx:]

print(f"Split completed: {len(train_folders)} Train, {len(val_folders)} Val, {len(test_folders)} Test")

print("Fitting dynamic scalers on training split...")
sample_1d, sample_2d = [], []
for folder in train_folders[:20]:
    s1 = pd.read_csv(os.path.join(train_path, folder, '1d_nodes_dynamic_all.csv')).drop(columns=['node_idx', 'timestep'])
    s2 = pd.read_csv(os.path.join(train_path, folder, '2d_nodes_dynamic_all.csv')).drop(columns=['node_idx', 'timestep'])
    sample_1d.append(s1)
    sample_2d.append(s2)

scaler_dyn_1d.fit(pd.concat(sample_1d).fillna(0))
scaler_dyn_2d.fit(pd.concat(sample_2d).fillna(0))

all_scalers = (scaler_1d_static, scaler_2d_static, scaler_dyn_1d, scaler_dyn_2d)

# --- 3. EDGE CONSTRUCTION ---
mapping_1d = {id: i for i, id in enumerate(df_1d_static['node_idx'])}
mapping_2d = {id: i for i, id in enumerate(df_2d_static['node_idx'])}

e1 = get_edge_index(os.path.join(train_path, '1d_edge_index.csv'), mapping_1d, mapping_1d)
e2 = get_edge_index(os.path.join(train_path, '2d_edge_index.csv'), mapping_2d, mapping_2d)
e12 = get_edge_index(os.path.join(train_path, '1d2d_connections.csv'), mapping_1d, mapping_2d, src_col='node_1d', dst_col='node_2d')
edges = (e1, e2, e12)

# --- 4. THE COMPLETE PROCESSING FUNCTION ---


# --- 5. EXECUTION ---
datasets = [
    (train_folders, '../Data/Models/Model_1/train/Processed/', "Train"),
    (val_folders, '../Data/Models/Model_1/val/Processed/', "Validation"),
    (test_folders, '../Data/Models/Model_1/internal_test/Processed/', "Internal_Test")
]

for folders, out_dir, tag in datasets:
    process_flood_directory(train_path, folders, out_dir, edges, all_scalers, tag)

Fitting static scalers...
Split completed: 54 Train, 7 Val, 7 Test
Fitting dynamic scalers on training split...


Processing Train:   0%|          | 0/54 [00:00<?, ?it/s]


--- Train Sample Verified: event_25 ---
Nodes: 1D=17, 2D=3716 | Timesteps=445
Edges: Pipe=16, Surface=7935, Exchange=16
Feature Vector Size: 1D=5

--- Train Final Report ---
Total: 54
Skipped Missing Val: 0
Skipped Physical Violation: 0
Skipped Shape Mismatch: 0
Skipped Empty Graph: 0
Success: 54


Processing Validation:   0%|          | 0/7 [00:00<?, ?it/s]


--- Validation Final Report ---
Total: 7
Skipped Missing Val: 0
Skipped Physical Violation: 0
Skipped Shape Mismatch: 0
Skipped Empty Graph: 0
Success: 7


Processing Internal_Test:   0%|          | 0/7 [00:00<?, ?it/s]


--- Internal_Test Final Report ---
Total: 7
Skipped Missing Val: 0
Skipped Physical Violation: 0
Skipped Shape Mismatch: 0
Skipped Empty Graph: 0
Success: 7


In [11]:
class FloodDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the pre-processed HeteroData object from disk
        return torch.load(self.file_paths[idx])

In [12]:

# 1. Collect all processed .pt file paths for each split
# Using glob to grab every file in the Processed directories we just created
train_files = glob.glob('../Data/Models/Model_1/train/Processed/*.pt')
val_files = glob.glob('../Data/Models/Model_1/val/Processed/*.pt')
test_files = glob.glob('../Data/Models/Model_1/internal_test/Processed/*.pt')

# 2. Initialize the Datasets
# Assuming your FloodDataset class is designed to take a list of file paths
train_ds = FloodDataset(train_files)
val_ds = FloodDataset(val_files)
test_ds = FloodDataset(test_files)

# 3. Create the Loaders
# shuffle=True for training to ensure the model doesn't learn the order of storms
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# Quick sanity check for your report
print(f"Loaders initialized:")
print(f" - Training Batches: {len(train_loader)}")
print(f" - Validation Batches: {len(val_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Loaders initialized:
 - Training Batches: 68
 - Validation Batches: 7
 - Test Batches: 7


### Baseline Model, Random Forest

In [13]:
def prepare_tabular_data(loader):
    """
    Flattens HeteroData objects into X (features) and y (targets).
    For the baseline, we predict the 'next' timestep for each node.
    """
    X_list, y_list = [], []
    
    for data in loader:
        # data['node1d'].x shape is [Timesteps, Nodes, Features]
        # We'll focus on 1D nodes for this example baseline
        x_1d = data['node1d'].x.numpy()
        
        # Create sliding window: Use time 't' to predict 't+1'
        # Features: [t], Target: [t+1, water_depth_index]
        # Assuming water depth is the last dynamic feature added
        X_frames = x_1d[:-1, :, :] # All steps except last
        y_frames = x_1d[1:, :, -1:] # All steps except first, depth only
        
        # Flatten [Timesteps * Nodes, Features]
        X_flat = X_frames.reshape(-1, X_frames.shape[-1])
        y_flat = y_frames.reshape(-1)
        
        X_list.append(X_flat)
        y_list.append(y_flat)
        
    return np.vstack(X_list), np.concatenate(y_list)

In [14]:
print("Flattening data for Random Forest...")
X_train, y_train = prepare_tabular_data(train_loader)
X_val, y_val = prepare_tabular_data(val_loader)

rf_baseline = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)

print("Training Random Forest Baseline...")
rf_baseline.fit(X_train, y_train)

# 3. Evaluate
y_pred = rf_baseline.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Baseline Validation RMSE: {rmse:.4f}")

Flattening data for Random Forest...


C:\Users\jaden\AppData\Local\Temp\ipykernel_24812\4200238874.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(self.file_paths[idx])


Training Random Forest Baseline...
Baseline Validation RMSE: 0.1450


### Primary Model

In [15]:
class FloodGNN(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        
        # 1. Spatial Layer (Same as before)
        self.conv = HeteroConv({
            ('node1d', 'pipe', 'node1d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'surface', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node1d', 'exchange', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'exchange', 'node1d'): SAGEConv((-1, -1), hidden_channels),
        }, aggr='sum')

        # 2. Temporal Layers
        self.gru_1d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)
        self.gru_2d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)

        # 3. Output Heads
        self.lin_1d = nn.Linear(hidden_channels, out_channels)
        self.lin_2d = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, h_dict=None):
            # Spatial pass
            h_spatial = self.conv(x_dict, edge_index_dict)
            h_spatial = {key: torch.relu(h) for key, h in h_spatial.items()}
            
            # Temporal pass
            # node1d
            x_1d = h_spatial['node1d'].unsqueeze(1) # [Nodes, 1, Hidden]
            h_1d, next_h1d = self.gru_1d(x_1d, h_dict['node1d'] if h_dict else None)
            
            # node2d
            x_2d = h_spatial['node2d'].unsqueeze(1) # [Nodes, 1, Hidden]
            h_2d, next_h2d = self.gru_2d(x_2d, h_dict['node2d'] if h_dict else None)

            # 3. Output pass
            # h_1d is [Nodes, 1, Hidden], so squeeze(1) gets us [Nodes, Hidden] for the Linear layer
            out_1d = self.lin_1d(h_1d.squeeze(1))
            out_2d = self.lin_2d(h_2d.squeeze(1))
            
            return out_1d, out_2d, {'node1d': next_h1d, 'node2d': next_h2d}

In [16]:
def calculate_dataset_std(loader):
    all_1d_levels = []
    all_2d_levels = []
    
    print("Calculating dataset statistics...")
    for batch in tqdm(loader):
        # Extract the target water level (last column of the feature matrix)
        # Shape of x: [Timesteps, Nodes, Features]
        y_1d = batch['node1d'].x[:, :, -1].flatten()
        y_2d = batch['node2d'].x[:, :, -1].flatten()
        
        all_1d_levels.append(y_1d.numpy())
        all_2d_levels.append(y_2d.numpy())
    
    # Concatenate all events into one giant array
    full_1d = np.concatenate(all_1d_levels)
    full_2d = np.concatenate(all_2d_levels)
    
    # Calculate Standard Deviation
    std_1d = np.std(full_1d)
    std_2d = np.std(full_2d)
    
    return std_1d, std_2d

In [17]:
std_1d, std_2d = calculate_dataset_std(train_loader)

# Create your own dictionary for the loss function
std_dev_dict = {
    (1, 1): std_1d, 
    (1, 2): std_2d,
    (2, 1): std_1d, # If Model 2 is trained on the same data, use the same std
    (2, 2): std_2d
}

print(f"\nCalculated 1D Std: {std_1d:.4f}")
print(f"Calculated 2D Std: {std_2d:.4f}")

Calculating dataset statistics...


  0%|          | 0/68 [00:00<?, ?it/s]

C:\Users\jaden\AppData\Local\Temp\ipykernel_24812\4200238874.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(self.file_paths[idx])



Calculated 1D Std: 0.9896
Calculated 2D Std: 0.9697


In [18]:


# 3. Standardized RMSE Loss
class StandardizedRMSELoss(nn.Module):
    def __init__(self, std_dev_dict):
        super().__init__()
        self.std_dev_dict = std_dev_dict
    
    def forward(self, pred_1d, pred_2d, target_1d, target_2d, model_id):
        # Get std devs for this model
        std_1d = self.std_dev_dict[(model_id, 1)]
        std_2d = self.std_dev_dict[(model_id, 2)]
        
        # Calculate RMSE for each node type
        rmse_1d = torch.sqrt(torch.mean((pred_1d - target_1d) ** 2) + 1e-6)
        rmse_2d = torch.sqrt(torch.mean((pred_2d - target_2d) ** 2) + 1e-6)
        
        # Standardize by dividing by std dev
        std_rmse_1d = rmse_1d / std_1d
        std_rmse_2d = rmse_2d / std_2d
        
        # Average across node types
        return (std_rmse_1d + std_rmse_2d) / 2

In [19]:
# Hardware check
print(f"Using device: {device}")
 
# Create two models
# Model 1: Baseline
model_1 = FloodGNN(hidden_channels=32, out_channels=1).to(device)

# Model 2: High-Capacity 
model_2 = FloodGNN(hidden_channels=128, out_channels=1).to(device)

# 5. Optimizers & Loss
optimizer_1 = optim.Adam(model_1.parameters(), lr=1e-4)
optimizer_2 = optim.Adam(model_2.parameters(), lr=1e-4)
criterion = StandardizedRMSELoss(std_dev_dict)


Using device: cuda


In [20]:
def train_event(model, event_data, optimizer, criterion, model_id, teacher_forcing_ratio=0.5):
    model.train()
    event_data = event_data.to(device)
    optimizer.zero_grad()
    
    num_t = event_data['node1d'].x.size(0)
    # Use a tensor for backprop, but a float for tracking
    accumulated_loss = 0 
    running_loss_val = 0.0
    
    h_dict = None 
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        loss = criterion(pred_1d, pred_2d, target_1d, target_2d, model_id)
        accumulated_loss = accumulated_loss + loss
        running_loss_val += loss.item() # .item() breaks the graph for tracking
        
        # Next input logic
        if random.random() < teacher_forcing_ratio:
            current_x_1d = event_data['node1d'].x[t]
            current_x_2d = event_data['node2d'].x[t]
        else:
            # Construct next input using predictions
            next_x_1d = event_data['node1d'].x[t].clone()
            next_x_1d[:, -1] = pred_1d.detach().squeeze()
            next_x_2d = event_data['node2d'].x[t].clone()
            next_x_2d[:, -1] = pred_2d.detach().squeeze()
            current_x_1d, current_x_2d = next_x_1d, next_x_2d

    # Backprop once per event
    accumulated_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    return running_loss_val / (num_t - 1)
def train_model(model, train_loader, val_loader, optimizer, criterion, model_id, num_epochs=20):
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
            # Decrease teacher forcing over time (Curriculum Learning)
            tf_ratio = max(0.1, 0.5 - (epoch * 0.05)) 
            loss = train_event(model, batch, optimizer, criterion, model_id, tf_ratio)
            total_train_loss += loss
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- Validation Phase ---
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                # Validation uses NO teacher forcing (0.0) to test real performance
                val_loss = validate_event(model, batch, criterion, model_id)
                total_val_loss += val_loss
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.6f} | Val Loss = {avg_val_loss:.6f}")

        # Save the best model
        if avg_val_loss == min(val_losses):
            torch.save(model.state_dict(), f'best_model_{model_id}.pth')
            
    return train_losses, val_losses

def validate_event(model, event_data, criterion, model_id):
    """Simplified version of train_event without backprop or teacher forcing"""
    h_dict = None
    num_t = event_data['node1d'].x.size(0)
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]
    total_loss = 0

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        total_loss += criterion(pred_1d, pred_2d, target_1d, target_2d, model_id).item()

        # Autoregressive step: always use predictions for validation
        next_x_1d = event_data['node1d'].x[t].clone()
        next_x_1d[:, -1] = pred_1d.squeeze()
        next_x_2d = event_data['node2d'].x[t].clone()
        next_x_2d[:, -1] = pred_2d.squeeze()
        current_x_1d, current_x_2d = next_x_1d, next_x_2d

    return total_loss / (num_t - 1)

In [ ]:
# Train Model 1 (Baseline)
print("Starting Training for Model 1...")
history_m1 = train_model(
    model=model_1, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    optimizer=optimizer_1, 
    criterion=criterion, 
    model_id=1, 
    num_epochs=2
)

# Train Model 2 (Enhanced)
print("\nStarting Training for Model 2...")
history_m2 = train_model(
    model=model_2, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    optimizer=optimizer_2, 
    criterion=criterion, 
    model_id=2, 
    num_epochs=2
)

Starting Training for Model 1...


Epoch 1/2 [Train]:   0%|          | 0/68 [00:00<?, ?it/s]

C:\Users\jaden\AppData\Local\Temp\ipykernel_24812\4200238874.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(self.file_paths[idx])


Epoch 1: Train Loss = 0.830054 | Val Loss = 0.858462


Epoch 2/2 [Train]:   0%|          | 0/68 [00:00<?, ?it/s]

Epoch 2: Train Loss = 0.774487 | Val Loss = 0.837636

Starting Training for Model 2...


Epoch 1/2 [Train]:   0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
def evaluate_and_predict(model, loader, model_name="Model"):
    model.eval()
    all_preds_1d, all_preds_2d = [], []
    all_actual_1d, all_actual_2d = [], []
    
    total_rmse_1d, total_rmse_2d = 0, 0
    event_count = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating {model_name}"):
            batch = batch.to(device)
            num_t = batch['node1d'].x.size(0)
            
            # Ground truth for comparison (excluding first step used as input)
            # Assuming water level is the last feature in your feature vector
            actual_1d = batch['node1d'].x[1:, :, -1].cpu().numpy()
            actual_2d = batch['node2d'].x[1:, :, -1].cpu().numpy()
            
            pred_1d_list, pred_2d_list = [], []
            h_dict = None
            
            # Autoregressive Prediction Loop
            curr_x1d, curr_x2d = batch['node1d'].x[0], batch['node2d'].x[0]
            
            for t in range(1, num_t):
                x_dict = {'node1d': curr_x1d, 'node2d': curr_x2d}
                p1d, p2d, h_dict = model(x_dict, batch.edge_index_dict, h_dict)
                
                pred_1d_list.append(p1d.cpu().numpy())
                pred_2d_list.append(p2d.cpu().numpy())
                
                # Update inputs for next step
                curr_x1d, curr_x2d = batch['node1d'].x[t], batch['node2d'].x[t]

            # Stack predictions for this event
            preds_1d = np.stack(pred_1d_list).squeeze()
            preds_2d = np.stack(pred_2d_list).squeeze()
            
            # Metrics Calculation
            total_rmse_1d += np.sqrt(np.mean((preds_1d - actual_1d)**2))
            total_rmse_2d += np.sqrt(np.mean((preds_2d - actual_2d)**2))
            
            all_preds_1d.append(preds_1d)
            all_preds_2d.append(preds_2d)
            all_actual_1d.append(actual_1d)
            all_actual_2d.append(actual_2d)
            event_count += 1

    # Aggregate Statistics
    avg_rmse_1d = total_rmse_1d / event_count
    avg_rmse_2d = total_rmse_2d / event_count
    
    print(f"\n--- {model_name} Quantitative Results ---")
    print(f"Average 1D (Sewer) RMSE: {avg_rmse_1d:.4f}")
    print(f"Average 2D (Surface) RMSE: {avg_rmse_2d:.4f}")
    
    return {
        'preds_1d': all_preds_1d, 'preds_2d': all_preds_2d,
        'actual_1d': all_actual_1d, 'actual_2d': all_actual_2d,
        'metrics': {'rmse_1d': avg_rmse_1d, 'rmse_2d': avg_rmse_2d}
    }


In [ ]:
# Run for both models
results_m1 = evaluate_and_predict(model_1, test_loader, "Model 1 (GNN-Baseline)")
results_m2 = evaluate_and_predict(model_2, test_loader, "Model 2 (Enhanced-GNN)")

In [ ]:


def plot_learning_curve(history, model_name):
    train_loss, val_loss = history
    plt.figure(figsize=(8, 5))
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title(f'Learning Curve: {model_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Standardized RMSE Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_learning_curve(history_m1, "Model 1")
plot_learning_curve(history_m2, "Model 2")

In [ ]:
# Load scalers and inverse-transform predictions back to original scale
scaler_dyn_1d = joblib.load('scaler_dyn_1d.pkl')
scaler_dyn_2d = joblib.load('scaler_dyn_2d.pkl')

# Get the water level column scaling parameters
water_level_idx = -1
scale_1d = scaler_dyn_1d.scale_[water_level_idx]
mean_1d = scaler_dyn_1d.mean_[water_level_idx]
scale_2d = scaler_dyn_2d.scale_[water_level_idx]
mean_2d = scaler_dyn_2d.mean_[water_level_idx]

all_predictions_1d_model1 = results_m1['preds_1d']
all_predictions_2d_model1 = results_m1['preds_2d']

all_predictions_1d_model2 = results_m2['preds_1d']
all_predictions_2d_model2 = results_m2['preds_2d']

# Inverse transform Model 1 predictions
all_predictions_1d_model1_original = [np.clip(pred * scale_1d + mean_1d, a_min=0, a_max=None) for pred in all_predictions_1d_model1]
all_predictions_2d_model1_original = [np.clip(pred * scale_2d + mean_2d, a_min=0, a_max=None) for pred in all_predictions_2d_model1]

# Inverse transform Model 2 predictions
all_predictions_1d_model2_original = [np.clip(pred * scale_1d + mean_1d, a_min=0, a_max=None) for pred in all_predictions_1d_model2]
all_predictions_2d_model2_original = [np.clip(pred * scale_2d + mean_2d, a_min=0, a_max=None) for pred in all_predictions_2d_model2]

print("Model 1 predictions inverse-transformed to original scale (clipped to non-negative)")
print(f"1D predictions range: {min([p.min() for p in all_predictions_1d_model1_original])} to {max([p.max() for p in all_predictions_1d_model1_original])}")
print(f"2D predictions range: {min([p.min() for p in all_predictions_2d_model1_original])} to {max([p.max() for p in all_predictions_2d_model1_original])}")

print("\nModel 2 predictions inverse-transformed to original scale (clipped to non-negative)")
print(f"1D predictions range: {min([p.min() for p in all_predictions_1d_model2_original])} to {max([p.max() for p in all_predictions_1d_model2_original])}")
print(f"2D predictions range: {min([p.min() for p in all_predictions_2d_model2_original])} to {max([p.max() for p in all_predictions_2d_model2_original])}")
